# Reproducing Tian et al. (2025), Figure 4, from dandiset 001559

Paper. Tian X, Zhang Y, Du H, Huang W, Bizimana LA, Nishimura N, Ai H (2025). An Enhanced Red
Bioluminescent Indicator for Responsive Detection of Physiological Calcium Dynamics in Cells and
Mice. ACS Sensors 10(8), 5826-5833.
[doi:10.1021/acssensors.5c01093](https://doi.org/10.1021/acssensors.5c01093). The claims file was
written from the version of record (open access, Europe PMC PMC12351538): main text, Methods,
legends and Discussion. The Supporting Information was not in that text; it holds nothing on
Figure 4.

Dataset. [DANDI:001559](https://dandiarchive.org/dandiset/001559/0.250902.2058), version
0.250902.2058. The paper cites no version of the dandiset (it does not name the dandiset at all;
the dandiset names the paper). This is the only published version and so the most recent one. The
draft has the same 35 assets and the same size (29.39 GB); it was last modified at publication
time, so the two do not differ in anything the analysis uses.

What the dandiset holds, as found below: one mouse (sub-582364L) and only image series. Its
description says it holds "all the raw data presented in Figure 4 of the paper, including the
python scripts used for analysis and for figure generation"; the raw data of Figure 4 are there
(for the one mouse of Figure 4D), the scripts are not. There are no stimulus tables: the stimulus
times come from a marker recorded as image channel 1, and nothing says which recording used which
electrode or current. Every other figure of the paper (in vitro, cells, neurons, the BLA
experiment of Figure 5) is outside the dandiset.

| Claim | Tested here |
|---|---|
| fig4b-expression-healthy | tested |
| fig4c-speckle-responsive-region | blocked, metadata insufficient (no stimulus times for the speckle movies) |
| fig4d-steady-baseline | tested |
| fig4d-evoked-increase | tested; the readings of the criterion disagree |
| fig4d-4ma-single-trials | tested, but depends on fig4d-evoked-increase |
| fig4d-electrode-timing | tested, but depends on fig4d-evoked-increase |
| fig5-bla-footshock-increase, fig5-brep-minimal, fig5-10hz-imaging, fig5-double-bric | blocked, data missing (Figure 5 is not in the dandiset) |
| fig1a-lysate-18fold, fig1b-mutations, fig1d-purified-response, fig1e-kd, lysate-brightness | blocked, data missing |
| fig2b-hela-5fold, fig2c-hek-3p5fold, fig3-neurons-7p3fold, cell-brightness, hek-spectrum, table-s1-fold-gains | blocked, data missing |

The outcomes, with the reasons, are in the attempt record,
`attempts/001559-tian-2025-01.yaml`.

In [ ]:
import hashlib
import itertools
import time
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage, stats

import ebric

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()
files = ebric.survey()
print(f"dandiset {ebric.DANDISET_ID} version {ebric.DANDISET_VERSION}: {len(files)} NWB files, "
      f"{files.size_gb.sum():.2f} GB, read in {time.time() - started:.0f} s")

## Data check

In [ ]:
files["date"] = files.session.str[:8]
print("Subjects:", files.subject.unique().tolist(), "; date of birth:", files.date_of_birth.unique().tolist())
print("\nFiles by recording day:")
print(files.groupby("date").agg(files=("path", "size"), sessions=("session", "nunique"),
                                 channels=("channel", lambda c: sorted(set(c.dropna().astype(int)))),
                                 shapes=("shape", lambda s: sorted(set(s))), dtypes=("dtype", lambda s: sorted(set(s))),
                                 rate_hz=("rate_hz", lambda r: f"{r.min():.4f} to {r.max():.4f}"),
                                 GB=("size_gb", "sum")).to_string())
print("\nPer session:")
print(files.groupby("session").agg(n_files=("path", "size"), channels=("channel", lambda c: sorted(set(c.dropna().astype(int)))),
                                    timing=("timing", lambda s: sorted(set(s))), start=("session_start", "first")).to_string())
print("\nDescriptions of the series:", sorted(files.description.unique()))
print("Indicator:", files.indicator.unique().tolist(), "; location:", files.location.unique().tolist(),
      "; emission wavelength:", files.emission_nm.unique().tolist())
print("Session descriptions:", files.session_description.unique().tolist())
print("Scan line rates (lines/s):", sorted(files.scan_line_rate.dropna().unique().round(3)))
gs = files.dropna(subset=["grid_spacing"]).copy()
gs["field_um"] = [g[0] * s[1] for g, s in zip(gs.grid_spacing, gs["shape"])]
print("\nStated pixel size and the field it implies:")
print(gs.groupby("date").agg(grid_spacing=("grid_spacing", "first"), unit=("grid_unit", "first"),
                              pixels=("shape", lambda s: s.iloc[0][1]), field_um=("field_um", "first")).to_string())

All 35 files are from one mouse, sub-582364L. The paper used six mice for this experiment
(Methods, "C57BL/6J mice (n = 6)") and shows one representative mouse in Figure 4D, so the
dandiset holds one of the six. The archive's summary says one subject, which matches.

Every file is a `TwoPhotonSeries` with unit "n.a.", indicator and location "unknown" and no
emission wavelength, and every session description is empty. Three kinds of recording are there,
by day: three 40 Hz camera movies on 2025-04-07; six four-channel ScanImage acquisitions on
2025-05-31 (1000 frames of 512 by 512 pixels at 4.228 frames/s, about 236 s each); and two
four-channel ScanImage stacks on 2025-06-09. Nothing in the files says what each is; the checks
below identify them from the data.

The stated pixel size, 0.028 or 0.014 micrometres, gives a field of 14.4 micrometres, about one
neuron. The 2025-06-09 stack of cells (shown under fig4b-expression-healthy) holds about a hundred
or more cell bodies in its field, and the wide-field stack shows a whole branching vessel
tree, so the stated pixel size is wrong (likely a unit or scale error). No spatial measure below
uses it.

In [ ]:
# The ScanImage series give either a rate or a timestamp array; check that the timestamps are regular.
rows = []
for a in ebric.assets():
    if "20250531" not in a.path and "20250609" not in a.path:
        continue
    f = ebric.open_asset(a)
    name = list(f["acquisition"])[0]
    s = f["acquisition"][name]
    if "timestamps" in s:
        ts = s["timestamps"][:]
        d = np.diff(ts)
        rows.append({"session": ebric.session_of(a.path), "series": name, "n": len(ts), "first": ts[0],
                     "dt_min": d.min(), "dt_max": d.max(), "dt_median": np.median(d)})
ts_tab = pd.DataFrame(rows)
print(ts_tab.groupby("session").agg(files=("series", "size"), n=("n", "first"), first=("first", "first"),
                                     dt_min=("dt_min", "min"), dt_max=("dt_max", "max"), dt_median=("dt_median", "median")).to_string())
print("\nRates stated in the other files (frames/s):")
print(files[files.timing == "rate"].groupby("session").rate_hz.agg(["min", "max"]).to_string())

Where a file has timestamps they start at 0 and are regular, with intervals equal to the inverse
of the rate the other files state for the same kind of acquisition (0.2365 s, 4.228 frames/s, on
2025-05-31). So frame times are frame index over rate in every file of that day. The four
channels of an acquisition are four files with the same session start time and the same frames,
so they share one clock by construction.

### The 2025-05-31 recordings: stimulus marker and bioluminescence channels

In [ ]:
t0 = time.time()
pmt = ebric.load_pmt("20250531")
print(f"read {sum(len(v) for v in pmt.values())} files of 2025-05-31 in {time.time() - t0:.0f} s")
sessions = sorted(pmt)
rows = []
for s in sessions:
    for c in sorted(pmt[s]):
        L = pmt[s][c]["lines"]
        rows.append({"session": s, "channel": c, "frames": L.shape[0], "lines": L.shape[1],
                     "min": L.min(), "p1": np.percentile(L, 1), "median": np.median(L), "p99": np.percentile(L, 99),
                     "max": L.max(), "frame_mean_sd": L.mean(1).std()})
levels = pd.DataFrame(rows)
print("Line means (mean of the 512 pixels of each line; arbitrary units):")
print(levels.round(2).to_string(index=False))

In [ ]:
rows = []
onsets = {}
for s in sessions:
    r = pmt[s][ebric.MARKER_CHANNEL]
    on, off, first_full = ebric.marker_onsets(r)
    onsets[s] = on
    L = r["lines"]
    fm = L.mean(1)
    rows.append({"session": s, "pulses": len(on),
                 "low level": np.median(fm[fm < ebric.MARKER_THRESHOLD]) if (fm < ebric.MARKER_THRESHOLD).any() else np.nan,
                 "high level": np.median(fm[fm >= ebric.MARKER_THRESHOLD]),
                 "largest spread of the lines within a frame": float((L.max(1) - L.min(1)).max()),
                 "frames between the levels (-1950 to -620)": int(((fm > -1950) & (fm < -620)).sum()),
                 "durations_s": np.round(off - on[:len(off)], 3).tolist(),
                 "onsets_s": np.round(on, 2).tolist(),
                 "intervals_s": np.round(np.diff(on), 2).tolist()})
marker = pd.DataFrame(rows)
for _, r in marker.iterrows():
    print(r.to_dict())
stim_sessions = [s for s in sessions if len(onsets[s])]
print("\nSessions with marker pulses:", stim_sessions)
print("Trials per session:", {s: len(onsets[s]) for s in stim_sessions}, "; total", sum(len(onsets[s]) for s in stim_sessions))
print("Shortest interval between onsets: %.2f s" % min(np.diff(onsets[s]).min() for s in stim_sessions))

Channel 1 is a stimulus marker: it sits at about -580 and drops to about -1990 in pulses. Its
value is the same on every line of a frame (the spread within a frame is printed), so it is the
frame's average of the marker, and the frames in which a pulse starts or ends lie between the two
levels, at most two per pulse. Placing each edge within its frame by that value
(`ebric.marker_onsets`) gives pulses of 4.985 to 4.992 s in every session. The Methods give
trains of 5 s ("Electrical stimulation (5 Hz, 5 s, 2 mA and 4 mA)"), so the pulses are read as the
stimulation periods; the files do not say so. The first two sessions (14:42 and 14:46) have no
pulses, the other four have 8, 8, 7 and 10, at irregular intervals of 12.3 to 56.5 s. Nothing in
the files says which of the two electrodes or which current (2 or 4 mA) each session or trial
used.

In [ ]:
# Which channels carry the bioluminescence, and do the stimulus markers share their clock?
# Contrast: mean response (0 to 5 s after onset minus the 5 s before) at the true onsets against
# the same onsets moved together by a random circular shift (1000 shifts per session).
rng_seed = 11
rows = []
for s in stim_sessions:
    for c in (2, 3, 4):
        t, x = ebric.binned(pmt[s][c])
        true = ebric.trial_table(t, x, onsets[s])["resp_a"].mean()
        null = ebric.shifted_null(t, x, onsets[s], n=1000, seed=rng_seed)
        rows.append({"session": s, "channel": c, "true": true, "null_mean": null.mean(), "null_sd": null.std(),
                     "z": (true - null.mean()) / null.std(), "p_shift": (np.sum(null >= true) + 1) / (len(null) + 1)})
clock = pd.DataFrame(rows)
print(clock.round(4).to_string(index=False))

Channels 3 and 4 increase after the marker onsets, far beyond the shifted onsets in the two
middle sessions (z above 5) and less so in the first and last (z 1.7 to 2.9); channel 2 changes
by about a tenth as much in the two middle sessions and not in the others, as a channel that
passes a little of the red light would. So the marker and the bioluminescence share a clock,
consistent with their being channels of one acquisition. The sizes per session are the subject
of fig4d-evoked-increase.

In [ ]:
# Recordings deposited twice: compare every pair of files of the same channel.
digest = {(s, c): hashlib.sha1(pmt[s][c]["lines"].tobytes()).hexdigest()[:12] for s in sessions for c in pmt[s]}
print("Distinct line-mean arrays:", len(set(digest.values())), "of", len(digest))
rows = []
for c in (2, 3, 4):
    for s1, s2 in itertools.combinations(sessions, 2):
        a, b = pmt[s1][c]["lines"].ravel(), pmt[s2][c]["lines"].ravel()
        rows.append({"channel": c, "pair": f"{s1[9:]} {s2[9:]}", "r": np.corrcoef(a, b)[0, 1]})
dup = pd.DataFrame(rows)
print("Largest correlation between two sessions of one channel: %.3f" % dup.r.abs().max())

No two files hold the same data, and no two sessions of a channel are near copies (the largest
correlation is printed above), so each recording is counted once.

### The 2025-06-09 stacks: which channel is the red one

In [ ]:
t0 = time.time()
jun = [a for a in ebric.assets() if "ses-20250609" in a.path]
def _proj(a):
    d = ebric.describe(a)
    m, fm = ebric.stack_projection(a, step=2)
    return (d["session"], d["channel"]), (m, fm)
with ThreadPoolExecutor(8) as ex:
    proj = dict(ex.map(_proj, jun))
print(f"read {len(proj)} stacks of 2025-06-09 in {time.time() - t0:.0f} s")
rows = []
for (s, c), (m, fm) in sorted(proj.items()):
    rows.append({"session": s, "channel": c, "frames": len(fm), "frame mean first": fm[0], "frame mean max": fm.max(),
                 "frame mean last": fm[-1], "projection p1": np.percentile(m, 1), "projection p99": np.percentile(m, 99)})
print(pd.DataFrame(rows).round(1).to_string(index=False))
for s in sorted({k[0] for k in proj}):
    g = proj[(s, 2)][0].ravel()
    print(s, "pixel correlation of the mean projections: ch2 with ch3 %.3f, ch2 with ch4 %.3f, ch3 with ch4 %.3f" % (
        np.corrcoef(g, proj[(s, 3)][0].ravel())[0, 1], np.corrcoef(g, proj[(s, 4)][0].ravel())[0, 1],
        np.corrcoef(proj[(s, 3)][0].ravel(), proj[(s, 4)][0].ravel())[0, 1]))

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for i, s in enumerate(sorted({k[0] for k in proj})):
    for c in range(1, 5):
        m = proj[(s, c)][0]
        lo, hi = np.percentile(m, [1, 99.5])
        ax[i, c - 1].imshow(m, cmap="gray", vmin=lo, vmax=hi)
        ax[i, c - 1].set_title(f"{s} channel {c}\nmean projection, shown {lo:.0f} to {hi:.0f}", fontsize=9)
        ax[i, c - 1].axis("off")
plt.tight_layout()
plt.show()
print("Each panel is clipped to its 1st and 99.5th percentiles (printed in its title); 1.5% of pixels lie outside.")

In the wide-field stack (14:22) channel 2 shows the vessels bright, as FITC-dextran labels them
(Figure 4B legend, "Blood vessels were labeled with FITC-dextran (green)"); channel 4 shows a
bright region with the vessels dark, and its pixels are anticorrelated with channel 2; channel 3
shows both the bright region and the vessels, and is positively correlated with channel 2.
Channel 4 is therefore the channel that sees the red eBRIC emission without the green dye, as a
645/65 filter would (Methods, "a 645/65 filter on photomultiplier tube"); channel 3 passes some
green as well. This identifies channel 4 as the 645/65 channel, consistent with the data but not
stated in them; the analyses use channel 4 and show channel 3 as a variant. The frame means of
both stacks rise and fall over the frames, consistent with z-stacks through the tissue. Channel 1
holds only scan noise.

### The 2025-04-07 movies

In [ ]:
spk = {}
t0 = time.time()
apr = [a for a in ebric.assets() if "ses-20250407" in a.path]
with ThreadPoolExecutor(3) as ex:
    for a, r in zip(apr, ex.map(lambda a: ebric.speckle_head(a, n_frames=400), apr)):
        spk[ebric.session_of(a.path)] = r
print(f"read the first 400 frames of each of {len(spk)} movies in {time.time() - t0:.0f} s")
for s, (K, M, rate) in spk.items():
    m = M.reshape(M.shape[0], -1).mean(1)
    k = K.reshape(K.shape[0], -1).mean(1)
    print(f"{s}: rate {rate} frames/s; frame mean intensity, first 8 frames {np.round(m[:8]).tolist()}")
    print(f"   mean intensity by frame index mod 4: {[round(float(m[p::4].mean())) for p in range(4)]}, "
          f"s.d. within each phase {[round(float(m[p::4].std()), 1) for p in range(4)]}")
    print(f"   mean block contrast K (8 x 8 pixels) by phase: {[round(float(k[p::4].mean()), 4) for p in range(4)]}")
dur = files[files.date == "20250407"].assign(seconds=lambda d: [s[0] / r for s, r in zip(d["shape"], d.rate_hz)])
print("\nDurations (s):", dur.seconds.round(1).tolist())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a_, (s, (K, M, rate)) in zip(ax, spk.items()):
    km = K[1::4].mean(0)
    lo, hi = np.percentile(km, [1, 99])
    a_.imshow(km, cmap="gray", vmin=lo, vmax=hi)
    a_.set_title(f"{s}: mean contrast K, phase 1\nshown {lo:.3f} to {hi:.3f}", fontsize=9)
    a_.axis("off")
plt.tight_layout()
plt.show()

The three movies of 2025-04-07 are labelled as two-photon imaging, but they are 40 Hz camera
movies whose frames cycle through four exposure levels (the frame mean takes four values in
turn, with little spread within each), and their speckle contrast maps show the vessel tree. This
is consistent with multi-exposure laser speckle imaging (the paper cites Rivera and Schaffer's
multi-exposure speckle method), that is, the speckle data of Figure 4C. They carry no stimulus
marker, no description and no times of stimulation.

## fig4b-expression-healthy

> Two-photon fluorescence shows robust eBRIC expression in neuronal somata and processes in the
> whisker barrel cortex, and the neurons look healthy and intact three months after surgery.

Criterion: reproduced when the two-photon fluorescence images show labelled cell bodies and
processes in the transduced region; the date relative to surgery is reported from the metadata if
given; health is not judged.

In [ ]:
m20 = proj[("20250609T145026", 4)][0]
wide = proj[("20250609T142205", 4)][0]
blobs = {}
for sig in (4, 6):   # Laplacian of Gaussian, sigma in pixels of the 512-pixel projection
    lg = -ndimage.gaussian_laplace(m20, sig)
    mad = 1.4826 * np.median(np.abs(lg - np.median(lg)))
    peaks = (lg == ndimage.maximum_filter(lg, size=2 * sig + 1)) & (lg > np.median(lg) + 5 * mad)
    yy, xx = np.where(peaks)
    blobs[sig] = (yy, xx)
    ring = []
    for y, x in zip(yy, xx):
        y0, y1, x0, x1 = max(0, y - 3 * sig), y + 3 * sig + 1, max(0, x - 3 * sig), x + 3 * sig + 1
        ring.append(m20[y, x] / np.median(m20[y0:y1, x0:x1]))
    print(f"sigma {sig} px: {len(yy)} bright compact objects; centre over local median, median {np.median(ring):.2f} "
          f"(range {np.min(ring):.2f} to {np.max(ring):.2f})")
pct = np.percentile(m20, [5, 25, 50, 75, 99])
print("Stack of cells, channel 4 projection percentiles 5, 25, 50, 75, 99:", pct.round(1).tolist())
print("Labelled tissue between cells (median) over the darkest 5%% (vessel shadows): %.2f" % (pct[2] / max(pct[0], 1e-6)))
pw = np.percentile(wide, [5, 50, 95, 99])
print("Wide field, channel 4 projection percentiles 5, 50, 95, 99:", pw.round(1).tolist())
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
lo, hi = np.percentile(wide, [1, 99.5]); ax[0].imshow(wide, cmap="magma", vmin=lo, vmax=hi)
ax[0].set_title(f"20250609T142205 channel 4 (wide field), {lo:.0f} to {hi:.0f}")
lo, hi = np.percentile(m20, [1, 99.7]); ax[1].imshow(m20, cmap="gray", vmin=lo, vmax=hi)
ax[1].scatter(blobs[6][1], blobs[6][0], s=40, facecolors="none", edgecolors="r", lw=0.6)
ax[1].set_title(f"20250609T145026 channel 4 (cells), {lo:.0f} to {hi:.0f}; circles: sigma 6 objects")
for a_ in ax: a_.axis("off")
plt.tight_layout(); plt.show()
dates = {"date of birth": "2024-10-29", "speckle movies": "2025-04-07", "bioluminescence": "2025-05-31", "two-photon stacks": "2025-06-09"}
print(dates)
print("Days from the first recording through the window (2025-04-07) to the stacks (2025-06-09):",
      (pd.Timestamp("2025-06-09") - pd.Timestamp("2025-04-07")).days)
print("Age at the stacks (days):", (pd.Timestamp("2025-06-09") - pd.Timestamp("2024-10-29")).days)

**Result: reproduced.** The channel 4 stack of 2025-06-09 shows bright, compact, cell-sized
objects (184 at sigma 4 px and 91 at sigma 6 px, their centres a median 1.70 and 2.03 times the
local median) in labelled tissue whose median is 3.55 times the darkest 5% of the field (the
vessel shadows), and in the wide-field stack the brightest 5% of pixels lie above 519 units
against a median of 17, a bounded bright region under the window. The cell-sized objects are
labelled somata; processes are not resolved one by one, and the labelling between the somata
(the neuropil) is the form they take at this resolution, judged from the image. The files give
no surgery date: the stacks were taken 63 days after the first recording through the window, so
at least two months after surgery, consistent with but not showing the paper's three months.

## fig4c-speckle-responsive-region

> Laser speckle contrast imaging shows that the region of viral expression responds with blood
> flow changes to electrical stimulation of the whisker pad.

Blocked. The speckle movies are present (checked above), but they carry no stimulus times or
marker, and nothing in the dandiset records when the whisker pad was stimulated during them, so
"during stimulation compared with before it" cannot be formed from the data. Finding the
stimulation periods from the flow signal itself would assume the response the claim tests.

## Conditions of the stimulated sessions

The four stimulated sessions are taken to be the four conditions of Figure 4D (two electrodes by
two currents), one condition per session: the paper reports exactly four, the dandiset holds the
raw data of Figure 4 for one mouse, and each session holds one series of 7 to 10 trials. Which
session is which is not recorded. The criteria of the dependent claims need the current; the
paper says the 4 mA condition gives the response strong enough for single events, so the
assumption adopted is that the two sessions with the larger mean response are the 4 mA ones.
Every other pairing is shown where a result depends on it.

In [ ]:
CH = 4   # channel 4 is the 645/65 channel (data check); channel 3 is shown as a variant
trials = {}
for s in stim_sessions:
    for c in ebric.BL_CHANNELS:
        t, x = ebric.binned(pmt[s][c])
        trials[(s, c)] = ebric.trial_table(t, x, onsets[s])
dt = float(np.median(np.diff(ebric.binned(pmt[stim_sessions[0]][CH])[0])))
print(f"bin {dt * 1000:.1f} ms; 1 s moving average = {ebric.smooth(np.zeros(10), dt)[1]} bins")
mean_resp = pd.Series({s: trials[(s, CH)].resp_a.mean() for s in stim_sessions})
print("Mean response, channel 4, window a, per session:", mean_resp.round(2).to_dict())
four_ma = tuple(sorted(mean_resp.sort_values().index[-2:]))
two_ma = tuple(s for s in stim_sessions if s not in four_ma)
print("Assumed 4 mA sessions:", four_ma, "; 2 mA sessions:", two_ma)
pairings = list(itertools.combinations(stim_sessions, 2))
print("All", len(pairings), "ways to choose the two 4 mA sessions are shown where they matter.")

## fig4d-steady-baseline

> Infusing DTZ at 25 uL/min through the tail vein gives a steady level of baseline
> bioluminescence during the recording.

Criterion: per recording, the baseline of each trial (mean of the 5 s before onset); reproduced
when the baseline changes from the first to the last trial by less than the mean 4 mA evoked
response. The criterion compares with the 4 mA response "in the same recording"; here a
recording holds one condition, so the 2 mA recordings have no 4 mA response of their own. The
comparison is made with the mean 4 mA response of the same channel (the two assumed 4 mA
sessions), in both response windows; the recording's own response is shown as a variant.

In [ ]:
rows = []
for c in ebric.BL_CHANNELS:
    ref = {k: np.mean([trials[(s, c)][f"resp_{k}"].mean() for s in four_ma]) for k in "ab"}
    for s in stim_sessions:
        tb = trials[(s, c)]
        drift = tb.baseline.iloc[-1] - tb.baseline.iloc[0]
        rows.append({"channel": c, "session": s, "trials": len(tb), "baseline first": tb.baseline.iloc[0],
                     "baseline last": tb.baseline.iloc[-1], "change": drift,
                     "baseline range": tb.baseline.max() - tb.baseline.min(),
                     "4 mA mean resp a": ref["a"], "4 mA mean resp b": ref["b"],
                     "|change| < 4 mA a": abs(drift) < ref["a"], "|change| < 4 mA b": abs(drift) < ref["b"],
                     "own resp a": tb.resp_a.mean(), "|change| < own a": abs(drift) < tb.resp_a.mean()})
steady = pd.DataFrame(rows)
print(steady.round(2).to_string(index=False))
print("\nUnder every pairing of 4 mA sessions (channel 4): recordings whose |change| is below the 4 mA mean response")
for p in pairings:
    for k in "ab":
        ref = np.mean([trials[(s, CH)][f"resp_{k}"].mean() for s in p])
        ok = [s[9:] for s in stim_sessions if abs(trials[(s, CH)].baseline.iloc[-1] - trials[(s, CH)].baseline.iloc[0]) < ref]
        print(f"  4 mA = {[s[9:] for s in p]}, window {k}: reference {ref:.2f}; passing {ok}")
# The baseline over the day, from the unstimulated sessions too (channel 4, mean of first and last 20 s)
rows = []
for s in sessions:
    t, x = ebric.binned(pmt[s][CH])
    rows.append({"session": s, "first 20 s": x[t < 20].mean(), "last 20 s": x[t > t[-1] - 20].mean(),
                 "change": x[t > t[-1] - 20].mean() - x[t < 20].mean()})
print("\nChannel 4 level within each session, all six sessions:")
print(pd.DataFrame(rows).round(2).to_string(index=False))
fig, ax = plt.subplots(1, len(stim_sessions), figsize=(16, 3.5), sharey=False)
for a_, s in zip(ax, stim_sessions):
    tb = trials[(s, CH)]
    a_.plot(tb.onset_s, tb.baseline, "o-")
    a_.set_title(f"{s[9:]}: range {tb.baseline.min():.1f} to {tb.baseline.max():.1f}", fontsize=9)
    a_.set_xlabel("onset (s)")
ax[0].set_ylabel("channel 4 baseline (a.u.)")
plt.tight_layout(); plt.show()

**Result: partially reproduced, resting on the assumed 4 mA sessions.** In three of the four
stimulated recordings the baseline changes from the first to the last trial by less than the
mean 4 mA response (at most 6.64 against 14.13 or more), in both channels and both windows. In
the first stimulated recording (14:50) the channel 4 baseline rises by 18.34 units (channel 3 by
17.53), more than the mean 4 mA response in either window (17.44 and 14.13), so there the
responses do not stand out from the drift by this criterion. The channel 4 level also rises
within the two unstimulated sessions before it (by 6.39 and 11.02) and from about 352 to about
402 over the day. With the 4 mA reference taken from the two weak sessions (2.49 or 2.02), the
14:55 recording fails as well.

## fig4d-evoked-increase

> Whisker pad stimulation (5 Hz for 5 s) at 4 mA and at 2 mA, through each of the two
> electrodes, gives clear increases in cortical eBRIC bioluminescence.

Criterion: per condition, the trial responses (mean in the response window minus the mean of the
5 s before onset) tested one-sided against zero, with windows (a) 0 to 5 s and (b) 0 to 10 s and
tests (1) Wilcoxon signed-rank and (2) one-sample t-test; an increase when the mean is positive
and p < 0.05; reproduced when all four conditions show an increase, the outcome being the one all
windows and tests give, and methods_underspecified when they differ. One mouse is in the
dandiset, so both readings of which mice are the same. With one condition per session, the
result does not depend on which session is which condition.

In [ ]:
rows = []
for c in ebric.BL_CHANNELS:
    for s in stim_sessions:
        tb = trials[(s, c)]
        for k in "ab":
            r = tb[f"resp_{k}"].to_numpy()
            pw = stats.wilcoxon(r, alternative="greater").pvalue
            pt = stats.ttest_1samp(r, 0, alternative="greater").pvalue
            rows.append({"channel": c, "session": s, "window": k, "trials": len(r),
                         "trials with full window": int(tb[f"complete_{k}"].sum()),
                         "mean": r.mean(), "sd": r.std(ddof=1), "d (mean/sd)": r.mean() / r.std(ddof=1),
                         "positive trials": int((r > 0).sum()), "p Wilcoxon": pw, "p t-test": pt,
                         "increase (1)": (r.mean() > 0) & (pw < 0.05), "increase (2)": (r.mean() > 0) & (pt < 0.05)})
evoked = pd.DataFrame(rows)
print(evoked.round(4).to_string(index=False))
print()
for c in ebric.BL_CHANNELS:
    for k in "ab":
        for test in ("(1)", "(2)"):
            e = evoked[(evoked.channel == c) & (evoked.window == k)]
            print(f"channel {c}, window {k}, test {test}: conditions with an increase {int(e['increase ' + test].sum())} of 4"
                  f" -> {'all four' if e['increase ' + test].all() else 'not all'}")

In [ ]:
# Trials whose 5 s baseline overlaps the 10 s window (b) of the trial before.
for s in stim_sessions:
    gaps = np.diff(onsets[s])
    print(s, "trials whose baseline starts within 10 s of the previous onset:", int((gaps < 15).sum()),
          "; intervals", np.round(gaps, 1).tolist())
fig, ax = plt.subplots(1, len(stim_sessions), figsize=(17, 3.8))
lims = []
for a_, s in zip(ax, stim_sessions):
    t, x = ebric.binned(pmt[s][CH])
    g, tr = ebric.event_average(t, x, onsets[s])
    base = tr[:, (g >= -5) & (g < 0)].mean(1, keepdims=True)
    y = tr - base
    for row in y:
        a_.plot(g, row, color="0.75", lw=0.6)
    a_.plot(g, y.mean(0), color="C0", lw=2)
    a_.axvspan(0, 5, color="0.9", zorder=0)
    a_.set_title(f"{s[9:]} ({len(tr)} of {len(onsets[s])} trials)", fontsize=9)
    a_.set_xlabel("time from onset (s)")
    lims.append((y.min(), y.max()))
ax[0].set_ylabel("channel 4, 1 s smoothed, minus baseline")
plt.tight_layout(); plt.show()
print("Range of the plotted traces per session:", [(round(float(a), 1), round(float(b), 1)) for a, b in lims])
print("Trials shown are those with 5 s before and 15 s after onset inside the recording.")

**Result: blocked, methods underspecified.** Two sessions respond strongly (channel 4, window a,
mean 15.93 and 18.95 units, every trial positive, Wilcoxon p = 0.0039 and 0.0078, t-test
p < 0.0001), and in channel 4 the 14:50 session shows a smaller increase (2.65 and 3.03 units)
significant under both windows and both tests. The 15:04 session decides the outcome: its mean
response is positive in both windows, but it is significant only by the t-test in window a
(p = 0.046; Wilcoxon p = 0.116; window b p = 0.246 and 0.211). Channel 3 gives the same split,
and there the 14:50 session also fails the Wilcoxon test in window a (p = 0.125). So all four
conditions show an increase under one reading of window and test and three of four (two of four
in one channel 3 reading) under the others, and the criterion makes the claim
methods_underspecified when the readings disagree. The 15:04 session's trials come at intervals
of 12.2 to 32.1 s, and the baselines of two of them start within 10 s of the previous onset.

## fig4d-4ma-single-trials

> At 4 mA the response is strong enough to detect individual stimulation events, in single trials
> without averaging.

Criterion: reading (a), single 5 s trials: a trial is detected when its response exceeds k times
the s.d. of its 1 s smoothed baseline, k = 2 or 3, in both windows; reproduced when more than half
of the 4 mA trials are detected under all of them. Reading (b), single pulses of the 5 Hz train, is
shown as a variant on the unsmoothed signal folded on the 0.2 s pulse period. The claim depends on
fig4d-evoked-increase.

In [ ]:
rows = []
for c in ebric.BL_CHANNELS:
    for s in stim_sessions:
        tb = trials[(s, c)]
        for k in "ab":
            for kk in (2, 3):
                det = tb[f"resp_{k}"] > kk * tb.baseline_sd_smoothed
                rows.append({"channel": c, "session": s, "assumed current": "4 mA" if s in four_ma else "2 mA",
                             "window": k, "k": kk, "detected": int(det.sum()), "trials": len(tb), "fraction": det.mean()})
single = pd.DataFrame(rows)
print(single.pivot_table(index=["channel", "session", "assumed current"], columns=["window", "k"], values="fraction").round(3).to_string())
print()
for c in ebric.BL_CHANNELS:
    s4 = single[(single.channel == c) & (single["assumed current"] == "4 mA")]
    pooled = s4.groupby(["window", "k"]).apply(lambda d: d.detected.sum() / d.trials.sum(), include_groups=False)
    print(f"channel {c}, assumed 4 mA sessions pooled, fraction detected:", pooled.round(3).to_dict(),
          "; min", round(float(pooled.min()), 3))
print("\nUnder every pairing of 4 mA sessions (channel 4): smallest pooled fraction over windows and k")
for p in pairings:
    s4 = single[(single.channel == CH) & single.session.isin(p)]
    pooled = s4.groupby(["window", "k"]).apply(lambda d: d.detected.sum() / d.trials.sum(), include_groups=False)
    print(f"  4 mA = {[s[9:] for s in p]}: min {pooled.min():.3f}, max {pooled.max():.3f}")

In [ ]:
# Variant (b): single pulses. Unsmoothed bins of 128 lines are too coarse for 0.2 s pulses, so the
# line signal is used (0.46 ms per line), each stimulation segment detrended, folded in 20 phase bins.
# Null: the same fold of the 5 s before each onset.
rng = np.random.default_rng(23)
rows = []
for c in ebric.BL_CHANNELS:
    for s in stim_sessions:
        x, t = pmt[s][c]["lines"].ravel(), pmt[s][c]["t"].ravel()
        prof, per = ebric.pulse_fold(t, x, onsets[s])
        pre, per0 = ebric.pulse_fold(t, x, onsets[s] - ebric.STIM_S)
        p2p, p2p0 = np.ptp(prof), np.ptp(pre)
        # trial-level test of the phase modulation: per-trial peak-to-peak, stimulation against before
        a, b = np.ptp(per, axis=1), np.ptp(per0, axis=1)
        rows.append({"channel": c, "session": s, "fold p2p during": p2p, "fold p2p before": p2p0,
                     "per-trial p2p during (median)": np.median(a), "before (median)": np.median(b),
                     "p (Wilcoxon, during > before)": stats.wilcoxon(a, b, alternative="greater").pvalue,
                     "phase of max (s)": np.argmax(prof) * 0.01})
pulses = pd.DataFrame(rows)
print(pulses.round(4).to_string(index=False))

**Result: blocked, because fig4d-evoked-increase is blocked (methods underspecified).** The test
itself ran: under the assumed current, every 4 mA trial in both sessions (15 of 15) exceeds 3
times the s.d. of its smoothed baseline in both windows and both channels, so more than half under
every reading. If the 4 mA pair includes one of the weak sessions, the pooled fraction is 0.53 to
0.75, still above half; if it is the two weak sessions, it is 0.28 to 0.50. The single-pulse
variant does not show responses to individual pulses: the folded modulation at the 0.2 s period
during stimulation is close to that of the 5 s before (channel 4, 0.31 against 0.27 and 0.47
against 0.37 in the two 4 mA sessions), and the per-trial comparison is not significant in
channel 4 (p 0.11 to 0.19).

## fig4d-electrode-timing

> The responses to the two electrode locations differ in timing; stimulation of region 1 gives a
> delayed peak, and region 2 an earlier rise after stimulation onset.

Criterion: per condition, the trial-averaged trace smoothed with a 1 s moving average, baseline
the 5 s before onset; time to peak is the time of the maximum from onset to 10 s; time of rise is
the first time after onset above half the peak. The dataset does not label the electrodes, so the
claim is tested as one electrode having both the later peak and the later rise, at either current.
The claim depends on fig4d-evoked-increase.

In [ ]:
rows = []
for c in ebric.BL_CHANNELS:
    for s in stim_sessions:
        t, x = ebric.binned(pmt[s][c])
        g, tr = ebric.event_average(t, x, onsets[s])
        pr = ebric.peak_and_rise(g, tr.mean(0))
        rows.append({"channel": c, "session": s, "assumed current": "4 mA" if s in four_ma else "2 mA",
                     "trials averaged": len(tr), **pr})
timing = pd.DataFrame(rows)
print(timing.round(3).to_string(index=False))

def later_both(c, pair):
    a, b = (timing[(timing.channel == c) & (timing.session == s)].iloc[0] for s in pair)
    if a.peak_time_s > b.peak_time_s and a.rise_time_s > b.rise_time_s:
        return pair[0]
    if b.peak_time_s > a.peak_time_s and b.rise_time_s > a.rise_time_s:
        return pair[1]
    return None

print()
for c in ebric.BL_CHANNELS:
    for name, pair in (("4 mA", four_ma), ("2 mA", two_ma)):
        r1 = later_both(c, pair)
        print(f"channel {c}, {name} {[s[9:] for s in pair]}: session with both the later peak and the later rise:",
              r1[9:] if r1 else "none")

# Bootstrap over trials: how often does the same session have both later times?
rng = np.random.default_rng(31)
print("\nBootstrap over trials (1000 resamples), fraction of resamples where each session has both later times:")
for c in ebric.BL_CHANNELS:
    for name, pair in (("4 mA", four_ma), ("2 mA", two_ma)):
        avgs = {}
        for s in pair:
            t, x = ebric.binned(pmt[s][c])
            avgs[s] = ebric.event_average(t, x, onsets[s])
        cnt = {pair[0]: 0, pair[1]: 0, None: 0}
        for _ in range(1000):
            pr = {}
            for s in pair:
                g, tr = avgs[s]
                pr[s] = ebric.peak_and_rise(g, tr[rng.integers(0, len(tr), len(tr))].mean(0))
            a, b = pr[pair[0]], pr[pair[1]]
            if a["peak_time_s"] > b["peak_time_s"] and a["rise_time_s"] > b["rise_time_s"]:
                cnt[pair[0]] += 1
            elif b["peak_time_s"] > a["peak_time_s"] and b["rise_time_s"] > a["rise_time_s"]:
                cnt[pair[1]] += 1
            else:
                cnt[None] += 1
        print(f"  channel {c}, {name}: {pair[0][9:]} {cnt[pair[0]] / 1000:.3f}, {pair[1][9:]} {cnt[pair[1]] / 1000:.3f}, neither {cnt[None] / 1000:.3f}")

print("\nUnder every pairing of the sessions into two currents (channel 4):")
for p in pairings:
    q = tuple(s for s in stim_sessions if s not in p)
    res = [later_both(CH, p), later_both(CH, q)]
    print(f"  {[s[9:] for s in p]} / {[s[9:] for s in q]}: later both -> {[r[9:] if r else None for r in res]}")

**Result: blocked, because fig4d-evoked-increase is blocked (methods underspecified).** The test
itself ran. At the assumed 2 mA the 14:50 session has both the later peak and the later rise in
both channels (channel 4, peak 4.42 against 3.60 s, rise 2.54 against 0.005 s), in 0.89 to 0.98
of bootstrap resamples; but the 15:04 average there is small (peak 2.54 units) and crosses half
its peak at onset. At the assumed 4 mA neither session has both later times: in channel 4 the
peaks fall in the same 59 ms bin (3.71 s), and in channel 3 the later peak goes with the earlier
rise. So the criterion's condition holds at one current, the weaker one, under the assumed
pairing; under the other pairings of sessions into currents some session also has both later
times at one current (printed above).

## Claims outside Figure 4

The dandiset holds the in vivo cortical recordings of Figure 4 only. The in vitro and cell data of
Figures 1 to 3 and S1 to S3, Table S1, and the awake BLA imaging of Figure 5 are not in it, and
the paper deposits no data elsewhere (its data statement names only the Addgene plasmids). These
claims are blocked, data missing: fig1a-lysate-18fold, fig1b-mutations, fig1d-purified-response,
fig1e-kd, lysate-brightness, fig2b-hela-5fold, fig2c-hek-3p5fold, fig3-neurons-7p3fold,
cell-brightness, hek-spectrum, table-s1-fold-gains, fig5-bla-footshock-increase,
fig5-brep-minimal, fig5-10hz-imaging and fig5-double-bric.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| fig4b-expression-healthy | reproduced |
| fig4c-speckle-responsive-region | blocked, metadata insufficient |
| fig4d-steady-baseline | partially reproduced (assumed) |
| fig4d-evoked-increase | blocked, methods underspecified |
| fig4d-4ma-single-trials | blocked, methods underspecified (depends on fig4d-evoked-increase) |
| fig4d-electrode-timing | blocked, methods underspecified (depends on fig4d-evoked-increase) |
| fifteen claims outside Figure 4 | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper gives no response window, baseline, test or
detection rule for Figure 4D; those of the claims file's criteria are used, each reading run. The
time base is bins of 128 scan lines (59 ms) of the frame signal. Channel 4 is taken as the 645/65
PMT and channel 3 shown as a variant. The four stimulated sessions are taken as the four
conditions, and the two with the larger responses as the 4 mA ones, an assumption the data cannot
check. One mouse of the paper's six is in the dandiset.

What was found about the dataset. It holds one mouse, as image series only. The stimulus times
are recorded only as an undescribed image channel, and no file says which electrode or current a
session used, which channel is which detector, or what the 2025-04-07 movies are (labelled
two-photon, they are consistent with multi-exposure speckle imaging and carry no stimulus times).
The stated pixel sizes (0.028 and 0.014 micrometres) cannot be right. The analysis scripts named
in the description are not in the dandiset.